<a href="https://colab.research.google.com/github/dogasumer/DSA210-Term-Project/blob/main/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mp-api pandas numpy matminer pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.8/308.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
import os
import pandas as pd
import numpy as np
from mp_api.client import MPRester
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

### Data from Materials Project

In [3]:
API_KEY = "jl33C95rdd7AunXvuoRDciMJDa4xOfNe"

with MPRester(API_KEY) as mpr:
    results = mpr.materials.summary.search(
        has_props=["magnetism"],
        fields=[
            "material_id",
            "formula_pretty",
            "elements",
            "nsites",
            "density",
            "formation_energy_per_atom",
            "band_gap",
            "total_magnetization",
            "total_magnetization_normalized_formula_units",
            "ordering",
            "symmetry"
        ],
        num_chunks=None
    )
    print(f"{len(results)} materials")

Retrieving SummaryDoc documents:   0%|          | 0/154879 [00:00<?, ?it/s]

154879 total materials fetched


##Build DataFrame & Sample

In [4]:
rows = []
for d in results:
    rows.append({
        "material_id": d.material_id,
        "formula": d.formula_pretty,
        "ordering": str(d.ordering),
        "band_gap": d.band_gap,
        "density": d.density,
        "nsites": d.nsites,
        "formation_energy_per_atom": d.formation_energy_per_atom,
        "total_magnetization": d.total_magnetization,
        "magnetization_per_fu": d.total_magnetization_normalized_formula_units,
        "n_elements": len(d.elements) if d.elements else None,
        "spacegroup": d.symmetry.symbol if d.symmetry else None,
        "crystal_system": d.symmetry.crystal_system.value if d.symmetry else None,
    })

df_full = pd.DataFrame(rows)
df_full = df_full[df_full['ordering'] != 'Unknown']

print("All available counts per ordering type:")
print(df_full["ordering"].value_counts())

All available counts per ordering type:
ordering
NM     83354
FM     56687
FiM    11345
AFM     3415
Name: count, dtype: int64


In [5]:
# Randomly sample ~5000 per ordering type (or all if fewer available)
sampled_groups = []
for otype in df_full["ordering"].unique():
    group = df_full[df_full["ordering"] == otype]
    n = min(5000, len(group))
    sampled_groups.append(group.sample(n=n, random_state=42))

df = pd.concat(sampled_groups).reset_index(drop=True)

print("Final sampled counts per ordering type:")
print(df["ordering"].value_counts())
print(f"\nTotal rows: {len(df)}")

Final sampled counts per ordering type:
ordering
NM     5000
FM     5000
FiM    5000
AFM    3415
Name: count, dtype: int64

Total rows: 18415


##Generate Compositional Features with Matminer (Magpie Preset)

In [6]:
# Convert formula strings to pymatgen Composition objects
df['composition'] = df['formula'].apply(
    lambda x: Composition(x) if pd.notnull(x) else None
)

# Drop rows where composition conversion failed
before = len(df)
df = df[df['composition'].notnull()].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with invalid formulas. Remaining: {len(df)}")

Converting formulas to Composition objects...
Dropped 0 rows with invalid formulas. Remaining: 18415


In [7]:
# Initialize Magpie featurizer
ep = ElementProperty.from_preset('magpie')

# Featurize — ignore_errors=True skips compounds that fail without crashing
df_featurized = ep.featurize_dataframe(df, col_id='composition', ignore_errors=True)

# Drop the pymatgen Composition column (not serializable to CSV)
df_featurized = df_featurized.drop(columns=['composition'])

print(f"Original columns: {len(df.columns)}")
print(f"Total columns after featurization: {len(df_featurized.columns)}")
print(f"New Magpie features added: {len(df_featurized.columns) - len(df.columns) + 1}")

Generating Magpie compositional features (this may take a few minutes)...


ElementProperty:   0%|          | 0/18415 [00:00<?, ?it/s]


Featurization complete.
Original columns: 13
Total columns after featurization: 144
New Magpie features added: 132


In [9]:
ep = ElementProperty.from_preset('magpie')
df_featurized = ep.featurize_dataframe(df, col_id='composition', ignore_errors=True)

# Keep only the most relevant Magpie columns
magpie_cols_to_keep = [
    'MagpieData mean Number',
    'MagpieData mean AtomicWeight',
    'MagpieData mean MeltingT',
    'MagpieData mean NUnfilled',
    'MagpieData mean NdUnfilled',
    'MagpieData mean NfUnfilled',
    'MagpieData mean Electronegativity',
    'MagpieData range Electronegativity',
    'MagpieData mean CovalentRadius',
    'MagpieData range CovalentRadius',
    'MagpieData mean GSmagmom',
    'MagpieData mean SpaceGroupNumber',
]

base_cols = ['material_id', 'formula', 'ordering', 'crystal_system', 'band_gap',
             'density', 'nsites', 'formation_energy_per_atom',
             'total_magnetization', 'magnetization_per_fu',
             'n_elements', 'spacegroup']

df_featurized = df_featurized[base_cols + magpie_cols_to_keep]

ElementProperty:   0%|          | 0/18415 [00:00<?, ?it/s]

##Missing Value Check

In [10]:
missing = df_featurized.isnull().sum()
missing_pct = (missing / len(df_featurized) * 100).round(2)

missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_%', ascending=False)

print("Columns with missing values:")
print(missing_df if len(missing_df) > 0 else "None")

Columns with missing values:
None


In [11]:
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

df.drop(columns=['composition'], errors='ignore').to_csv('data/raw/magnetic_materials_raw.csv', index=False)
df_featurized.to_csv('data/processed/materials_featurized.csv', index=False)

print(f"\nFinal dataset shape: {df_featurized.shape}")
print("\nFirst 5 rows (base columns only):")
base_cols = ['material_id', 'formula', 'ordering', 'crystal_system', 'band_gap',
             'density', 'nsites', 'formation_energy_per_atom', 'magnetization_per_fu']
print(df_featurized[base_cols].head())


Final dataset shape: (18415, 24)

First 5 rows (base columns only):
  material_id       formula ordering crystal_system  band_gap    density  \
0  mp-2226142  Ba2Mg(TiS3)2       NM       Trigonal    0.1992   3.615533   
1   mp-571341   InHg6As4Cl7       NM          Cubic    1.4202   6.436632   
2    mp-21267      Yb2InPd2       NM     Tetragonal    0.0000  10.709802   
3    mp-16958       LaNi5P3       NM   Orthorhombic    0.0000   7.254208   
4    mp-30011        SbKrF7       NM     Monoclinic    2.4093   3.876690   

   nsites  formation_energy_per_atom  magnetization_per_fu  
0      11                  -1.689290          9.000000e-07  
1      72                  -0.735864          0.000000e+00  
2      10                  -0.966467          0.000000e+00  
3      18                  -0.810871          1.945000e-05  
4      36                  -1.875156          0.000000e+00  
